# Hybrid GNN–Solver for AC Optimal Power Flow

**Physics-informed graph neural networks as warm-starters for Newton-Raphson**

The idea here is straightforward: instead of letting Newton-Raphson start from a flat profile every time (V=1, θ=0), train a GNN to predict a reasonable operating point from the network topology and load conditions, then use that as the initial guess. The GNN doesn't need to solve OPF perfectly - it just needs to get close enough that the solver does less work.

The pipeline goes: generate AC-OPF solutions across perturbed load scenarios → build a graph dataset → train three GNN architectures (GCN, GraphSAGE, GAT) with a physics-penalised loss → plug predictions into pandapower's NR solver → measure iteration counts, feasibility, and runtime.

In [ ]:
!pip install -q pandapower>=2.13.0 pypower pyyaml tqdm joblib
print(torch.__version__)
print(torch.version.cuda)
!pip install -q torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html

In [ ]:
import contextlib, copy, json, logging, os, random, signal, time, warnings, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

import pandapower as pp
import pandapower.networks as pn

from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATv2Conv
from torch_geometric.utils import add_self_loops

warnings.filterwarnings('ignore')
logging.getLogger('pandapower').setLevel(logging.ERROR)
logging.getLogger('numba').setLevel(logging.ERROR)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')

In [ ]:
CONFIG = {
    'seed': 42,

    'data_dir': Path('/content/drive/MyDrive/data'),
    'checkpoint_dir': Path('checkpoints'),
    'results_dir': Path('results'),
    'plots_dir': Path('results/plots'),

    # IEEE 14-, 30-, and 118-bus systems from pandapower's built-in cases. Larger cases (case57, case145, case300) were tried but dropped - 
    # AC-OPF doesn't converge reliably under Gaussian load perturbations at that scale. 
    'cases': ['case14', 'case30', 'case118'],

    # Each scenario = one perturbed load profile + its OPF solution.
    'scenarios_per_case': 500,

    # Gaussian noise on base Pd/Qd, clipped at max_noise_factor.
    'load_noise_std': 0.25,
    'max_noise_factor': 0.40,

    # Per-scenario OPF timeout. Anything that times out gets dropped from the dataset.
    'opf_timeout_sec': 45,

    'split': (0.80, 0.10, 0.10),  # train / val / test split

    # Node features: Pd, Qd, Vmin, Vmax, bus type (PQ / PV / slack)
    'node_feature_dim': 7,
    # Edge features: r, x, b, thermal limit - all normalised
    'edge_feature_dim': 4,
    # Targets: vm_pu, va_norm, p_gen_norm
    'target_dim': 3,

    'hidden_dim': 128,
    'num_layers': 3,
    'dropout': 0.10,
    'gat_heads':  4,

    'epochs': 150,
    'lr': 1e-3,
    'weight_decay': 1e-5,
    'batch_size': 32,
    'patience': 15,   # early-stopping patience on val loss
    'grad_clip': 1.0,

    # Total loss = MSE + lambda_physics * DC_balance_penalty. Lambda=0.1. Ablation study is a future work.
    'lambda_physics': 0.10,

    'max_nr_iterations': 50,
    'nr_tolerance': 1e-8,

    'v_min_pu': 0.90,
    'v_max_pu': 1.10,
    'line_overload_pct': 100.0,
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])

for key in ('data_dir', 'checkpoint_dir', 'results_dir', 'plots_dir'):
    CONFIG[key].mkdir(parents=True, exist_ok=True)

print(f"Cases: {CONFIG['cases']}")
print(f"Scenarios per case: {CONFIG['scenarios_per_case']}")
print(f"Physics lambda: {CONFIG['lambda_physics']}")

## Data Generation

For each IEEE case:
1. We load the base network from `pandapower.networks`
2. Perturb each bus's Pd / Qd with Gaussian noise (clipped at ±40%)
3. Run AC-OPF via `pp.runopp()` - this is the ground truth
4. Extract 7-dim node features and 4-dim edge features from the converged solution
5. Save the required ground truths: `vm_pu`, `va_degree`, `p_gen`

In [ ]:
class _OPFTimeout(Exception): pass

@contextlib.contextmanager
def time_limit(seconds):
    def _handler(s, f): raise _OPFTimeout
    old = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)

In [ ]:
CASE_LOADERS = {'case14': pn.case14, 'case30': pn.case30, 'case118': pn.case118}

def ensure_costs(net):
    have_gen = set()
    if len(net.poly_cost) > 0:
        have_gen.update(net.poly_cost[net.poly_cost.et == 'gen'].element.tolist())
    for idx, row in net.gen.iterrows():
        if idx in have_gen: continue
        pmax = float(row.get('max_p_mw', row.get('p_mw', 100.0)))
        if np.isnan(pmax) or pmax <= 0: pmax = 100.0
        pp.create_poly_cost(net, idx, 'gen', cp2_eur_per_mw2=0.01, cp1_eur_per_mw=2.0, cp0_eur=50.0)

    have_eg = set()
    if len(net.poly_cost) > 0:
        have_eg.update(net.poly_cost[net.poly_cost.et == 'ext_grid'].element.tolist())
    for idx in net.ext_grid.index:
        if idx in have_eg: continue
        pp.create_poly_cost(net, idx, 'ext_grid', cp2_eur_per_mw2=0.02, cp1_eur_per_mw=2.5, cp0_eur=60.0)

def load_base_network(case_name):
    network = CASE_LOADERS[case_name]()
    ensure_costs(network)
    return network

In [ ]:
def sample_scenario(net_base, rng, noise_std, max_factor):
    net = copy.deepcopy(net_base)
    if len(net.load) == 0:
        return net
    bp = net.load.p_mw.values.copy()
    bq = net.load.q_mvar.values.copy()
    np = np.clip(rng.normal(0, noise_std, len(bp)), -max_factor, max_factor)
    nq = np.clip(rng.normal(0, noise_std, len(bq)), -max_factor, max_factor)
    net.load.p_mw   = np.maximum(0.0, bp * (1 + np))   # loads stay non-negative
    net.load.q_mvar = bq * (1 + nq)
    return net

In [ ]:
def extract_features(net):
    """
    Pull node features, edge features, and targets from a converged OPF solution.

    Returns a dict:
        bus_features  : (N, 7) - [Pd_norm, Qd_norm, Vmin, Vmax, type_PQ, type_PV, type_slack]
        edge_index    : (2, E) - bidirectional, so each undirected line appears twice
        edge_features : (E, 4) - [r_norm, x_norm, bnorm, thermal_norm]
        targets       : (N, 3) - [vm_pu, va_norm, p_gen_norm]
        meta          : dict   - bookkeeping (n_bus, n_line, n_gen, obj_cost, total_load_mw)
    """
    bmap = {bid: i for i, bid in enumerate(net.bus.index)}
    N = len(net.bus)

    # Node features
    Pd = np.zeros(N, np.float32)
    Qd = np.zeros(N, np.float32)
    for _, r in net.load.iterrows():
        i = bmap[r.bus]
        Pd[i] += float(r.p_mw)
        Qd[i] += float(r.q_mvar)

    Vmin = np.array([float(net.bus.at[b, 'min_vm_pu']) if 'min_vm_pu' in net.bus.columns else 0.9 for b in net.bus.index], dtype=np.float32)
    Vmax = np.array([float(net.bus.at[b, 'max_vm_pu']) if 'max_vm_pu' in net.bus.columns else 1.1 for b in net.bus.index], dtype=np.float32)

    btype = np.zeros((N, 3), np.float32)   # one-hot: [PQ, PV, slack]
    slack_buses = set(net.ext_grid.bus.tolist())
    pv_buses = set(net.gen.bus.tolist())
    for bid, i in bmap.items():
        if bid in slack_buses:  
            btype[i, 2] = 1.0
        elif bid in pv_buses:   
            btype[i, 1] = 1.0
        else:                   
            btype[i, 0] = 1.0

    total_load = Pd.sum() + 1e-6
    bus_features = np.column_stack([Pd / total_load, Qd / total_load, Vmin, Vmax, btype]).astype(np.float32)

    # Edge features
    fi, ti, rv, xv, bv, thv = [], [], [], [], [], []

    for _, lr in net.line.iterrows():
        f, t = bmap[lr.from_bus], bmap[lr.to_bus]
        r = float(lr.r_ohm_per_km) * float(lr.length_km)
        x = float(lr.x_ohm_per_km) * float(lr.length_km)
        b = float(lr.get('c_nf_per_km', 0.0)) * float(lr.length_km)
        th = float(lr.get('max_i_ka', 1.0))
        for f_, t_ in [(f, t), (t, f)]:   # bidirectional
            fi.append(f_); ti.append(t_)
            rv.append(r);  xv.append(x); bv.append(b); thv.append(th)

    if len(net.trafo) > 0:
        for _, tr in net.trafo.iterrows():
            f, t = bmap[tr.hv_bus], bmap[tr.lv_bus]
            vn = float(net.bus.at[tr.hv_bus, 'vn_kv'])
            sn = float(tr.sn_mva) if tr.sn_mva > 0 else 100.0
            vk = float(tr.vk_percent) / 100.0
            r_t = float(tr.vkr_percent) / 100.0 * vn**2 / sn
            x_t = np.sqrt(max(vk**2 - r_t**2, 0.0)) * vn**2 / sn
            for f_, t_ in [(f, t), (t, f)]:
                fi.append(f_); ti.append(t_)
                rv.append(r_t); xv.append(x_t); bv.append(0.0); thv.append(sn)

    edge_index = np.array([fi, ti], dtype=np.int64)
    r_a, x_a = np.array(rv, np.float32), np.array(xv, np.float32)
    ba, th_a = np.array(bv, np.float32), np.array(thv, np.float32)
    # Normalisation of all edge features in each case.
    r_a /= r_a.max()  + 1e-6
    x_a /= x_a.max()  + 1e-6
    ba /= ba.max()  + 1e-6
    th_a /= th_a.max() + 1e-6
    edge_features = np.column_stack([r_a, x_a, ba, th_a]).astype(np.float32)

    # Targets
    vm_pu = np.array([float(net.res_bus.at[b, 'vm_pu']) for b in net.bus.index], np.float32)
    va_deg = np.array([float(net.res_bus.at[b, 'va_degree']) for b in net.bus.index], np.float32)
    p_gen = np.zeros(N, np.float32)
    for i, gr in net.res_gen.iterrows():
        p_gen[bmap[net.gen.at[gr.name, 'bus']]] += float(gr.p_mw)
    for i, eg in net.res_ext_grid.iterrows():
        p_gen[bmap[net.ext_grid.at[eg.name, 'bus']]] += float(eg.p_mw)

    va_norm = (va_deg / 180.0).astype(np.float32)
    p_gen_norm = (p_gen / (np.abs(p_gen).max() + 1e-6)).astype(np.float32)
    targets = np.column_stack([vm_pu, va_norm, p_gen_norm]).astype(np.float32)

    cost = float(net.res_cost) if hasattr(net, 'res_cost') else 0.0
    meta = {'n_bus': N, 
            'n_line': len(net.line), 
            'n_gen': len(net.gen), 
            'obj_cost': cost, 
            'total_load_mw': float(Pd.sum())
        }

    return dict(bus_features=bus_features, edge_index=edge_index, edge_features=edge_features, targets=targets, meta=meta)

In [ ]:
def generate_case(case_name, cfg):
    out_dir = cfg['data_dir'] / case_name
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(cfg['seed'])
    n = cfg['scenarios_per_case']
    timeout = cfg['opf_timeout_sec']
    std = cfg['load_noise_std']
    maxf = cfg['max_noise_factor']

    try:
        net_base = load_base_network(case_name)
    except Exception as e:
        print(f'Could not load {case_name}: {e}')
        return {'case': case_name, 'generated': 0, 'skipped': 0}

    nb = len(net_base.bus)
    nl = len(net_base.line)
    gen_count = skipped = 0
    costs = []

    pbar = tqdm(range(n), desc=f'{case_name} ({nb} buses)', leave=True)
    for i in pbar:
        out_path = out_dir / f'scenario_{i:04d}.npz'
        if out_path.exists():
            gen_count += 1
            continue

        net_s = sample_scenario(net_base, rng, std, maxf)

        try:
            with time_limit(timeout):
                pp.runopp(net_s, verbose=False, suppress_warnings=True, numba=False)
        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f'[OPF ERROR] scenario {i}: {type(e).__name__}: {e}')
            pbar.set_postfix({'ok': gen_count, 'skip': skipped})
            continue

        if not getattr(net_s, 'OPF_converged', False):
            skipped += 1
            if skipped <= 10:
                print(f'[NOT CONVERGED] scenario {i}')
            pbar.set_postfix({'ok': gen_count, 'skip': skipped})
            continue

        try:
            res = extract_features(net_s)
        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f'[FEATURE ERROR] scenario {i}: {type(e).__name__}: {e}')
            pbar.set_postfix({'ok': gen_count, 'skip': skipped})
            continue

        np.savez_compressed(
            out_path,
            bus_features = res['bus_features'],
            edge_index = res['edge_index'],
            edge_features = res['edge_features'],
            targets = res['targets'],
            meta = np.array([json.dumps(res['meta'])]),
        )
        costs.append(res['meta']['obj_cost'])
        gen_count += 1
        pbar.set_postfix({'ok': gen_count, 'skip': skipped})

    return {
        'case': case_name, 'n_bus': nb, 'n_line': nl,
        'generated': gen_count, 'skipped': skipped,
        'mean_cost': float(np.mean(costs)) if costs else 0.0
    }

def generate_dataset(cfg):
    print('Generating dataset via pandapower runopp()')
    summaries = []
    t0 = time.time()
    for case in cfg['cases']:
        s = generate_case(case, cfg)
        summaries.append(s)
        print(f"  {s['case']:10s}  generated={s['generated']:4d}  skipped={s['skipped']:4d}")
    df = pd.DataFrame(summaries)
    df.to_csv(cfg['data_dir'] / 'generation_summary.csv', index=False)
    print(f'Total: {df.generated.sum()} scenarios in {time.time()-t0:.0f}s')
    return df

# Only needs to be executed once. If the .npz files already exist, we can move on to the next cell.
gen_summary = generate_dataset(CONFIG)
gen_summary

## Dataset

Wraps the `.npz` files into a `torch_geometric.data.Dataset`. The 80/10/10 split is applied **per case** so all three network sizes appear in every partition.

In [ ]:
class OPFDataset(Dataset):
    # Each item: one OPF scenario as a PyG Data object.
    # .x (N, 7)  node features
    # .edge_index (2, E)
    # .edge_attr (E, 4)  edge features
    # .y (N, 3)  targets: [vm_pu, va_norm, p_gen_norm]
    def __init__(self, file_list):
        super().__init__()
        self.files = file_list

    def len(self): 
        return len(self.files)

    def get(self, idx):
        p = self.files[idx]
        d = np.load(p, allow_pickle=True)
        x = torch.tensor(d['bus_features'], dtype=torch.float32)
        ei = torch.tensor(d['edge_index'], dtype=torch.long)
        ea = torch.tensor(d['edge_features'], dtype=torch.float32)
        y = torch.tensor(d['targets'], dtype=torch.float32)
        meta = json.loads(str(d['meta'][0]))
        return Data(x=x, edge_index=ei, edge_attr=ea, y=y, case_name=p.parent.name, n_bus=meta['n_bus'], npz_path=str(p))

def make_splits(cfg):
    rng = np.random.default_rng(cfg['seed'])
    tr_f, va_f, te_f = [], [], []
    t_frac, v_frac, _ = cfg['split']

    for case in cfg['cases']:
        files = sorted((cfg['data_dir'] / case).glob('scenario_*.npz'))
        if len(files) == 0:
            print(f'No files for {case} - skipping')
            continue
        files = np.array(files)
        idx = rng.permutation(len(files))
        n_tr = int(len(idx) * t_frac)
        n_va = int(len(idx) * v_frac)
        tr_f += files[idx[:n_tr]].tolist()
        va_f += files[idx[n_tr:n_tr + n_va]].tolist()
        te_f += files[idx[n_tr + n_va:]].tolist()

    rng.shuffle(tr_f); rng.shuffle(va_f); rng.shuffle(te_f)
    print(f'Split: train={len(tr_f)} | val={len(va_f)} | test={len(te_f)}')
    return OPFDataset(tr_f), OPFDataset(va_f), OPFDataset(te_f)

train_ds, val_ds, test_ds = make_splits(CONFIG)
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

## The Three GNN Architectures

Three architectures, compared under identical training conditions. All three share the same depth (3 layers), hidden width (128), residual connections, dropout (0.1), and MLP decoder. The only thing changing between models is how neighbourhood information is aggregated. That way the accuracy comparison reflects the message-passing mechanism, not architecture differences elsewhere.

In [ ]:
# Shared decoder for all three models - isolates the comparison to the encoder.
class MLPDecoder(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),
            nn.ReLU(),
            nn.Dropout(CONFIG['dropout']),
            nn.Linear(in_dim // 2, out_dim),
        )
    def forward(self, x): return self.net(x)

class GCNModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, n_layers, dropout):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.skip = nn.ModuleList()
        dims = [in_dim] + [hidden] * n_layers
        for i in range(n_layers):
            self.convs.append(GCNConv(dims[i], dims[i + 1]))
            self.norms.append(nn.LayerNorm(dims[i + 1]))
            self.skip.append(
                nn.Linear(dims[i], dims[i + 1], bias=False)
                if dims[i] != dims[i + 1] else nn.Identity()
            )
        self.decoder = MLPDecoder(hidden, out_dim)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        for conv, norm, skip in zip(self.convs, self.norms, self.skip):
            h = F.dropout(F.relu(norm(conv(x, ei))), p=self.dropout, training=self.training)
            x = h + skip(x)
        return self.decoder(x)

class GraphSAGEModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, n_layers, dropout):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.skip = nn.ModuleList()
        dims = [in_dim] + [hidden] * n_layers
        for i in range(n_layers):
            self.convs.append(SAGEConv(dims[i], dims[i + 1]))
            self.norms.append(nn.LayerNorm(dims[i + 1]))
            self.skip.append(
                nn.Linear(dims[i], dims[i + 1], bias=False)
                if dims[i] != dims[i + 1] else nn.Identity()
            )
        self.decoder = MLPDecoder(hidden, out_dim)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        for conv, norm, skip in zip(self.convs, self.norms, self.skip):
            h = F.dropout(F.relu(norm(conv(x, ei))), p=self.dropout, training=self.training)
            x = h + skip(x)
        return self.decoder(x)

class GATModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, n_layers, heads, dropout, edge_dim=4):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.skip = nn.ModuleList()
        dims = [in_dim] + [hidden] * n_layers
        for i in range(n_layers):
            self.convs.append(GATv2Conv(dims[i], dims[i + 1], heads=heads, edge_dim=edge_dim, concat=False, dropout=dropout))
            self.norms.append(nn.LayerNorm(dims[i + 1]))
            self.skip.append(
                nn.Linear(dims[i], dims[i + 1], bias=False)
                if dims[i] != dims[i + 1] else nn.Identity()
            )
        self.decoder = MLPDecoder(hidden, out_dim)

    def forward(self, data):
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        for conv, norm, skip in zip(self.convs, self.norms, self.skip):
            h = F.dropout(F.relu(norm(conv(x, ei, edge_attr=ea))), p=self.dropout, training=self.training)
            x = h + skip(x)
        return self.decoder(x)

def build_model(name, cfg):
    in_d = cfg['node_feature_dim']
    hid = cfg['hidden_dim']
    out = cfg['target_dim']
    nl = cfg['num_layers']
    drop = cfg['dropout']
    if name == 'GCN':
        return GCNModel(in_d, hid, out, nl, drop)
    elif name == 'GraphSAGE':
        return GraphSAGEModel(in_d, hid, out, nl, drop)
    elif name == 'GAT':
        return GATModel(in_d, hid, out, nl, cfg['gat_heads'], drop, edge_dim=cfg['edge_feature_dim'])
    else:
        raise ValueError(f'Unknown model: {name}')

print('Trainable parameters:')
for name in ['GCN', 'GraphSAGE', 'GAT']:
    m = build_model(name, CONFIG)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f' {name}: {n:,}')

## Physics-Informed Loss

Total loss = supervised MSE + λ · DC power-balance penalty

The physics term enforces Kirchhoff's current law at every non-slack bus via the susceptance matrix:

$$\mathcal{L}_{\text{physics}} = \| \mathbf{B}\,\hat{\boldsymbol{\theta}} - \hat{\mathbf{P}}_{\text{net}} \|_2^2$$

where **B** is assembled from predicted reactances, $\hat{\theta}$ are predicted voltage angles, and $\hat{P}_{\text{net}}$ is predicted net injection. The slack bus is excluded.

Setting λ=0.1 keeps this as a soft regulariser rather than a hard constraint. The main supervised signal still comes from the OPF labels; the physics penalty just nudges predictions toward solutions that would actually be feasible. A more comprehensive ablation study to study the affects of lambda on the training and results is a future consideration, not a part of the current work.

In [ ]:
def build_susceptance_matrix(edge_index, edge_attr, n_bus, device):
    # Builds dense DC susceptance matrix B from normalised reactances (edge_attr col 1).
    B = torch.zeros(n_bus, n_bus, device=device)
    f = edge_index[0]
    t = edge_index[1]
    x = edge_attr[:, 1].clamp(min=1e-6)
    b = 1.0 / x

    mask = f < t
    f_, t_, b = f[mask], t[mask], b[mask]

    B.index_put_((f_, t_), -b, accumulate=True)
    B.index_put_((t_, f_), -b, accumulate=True)
    B.index_put_((f_, f_), b, accumulate=True)
    B.index_put_((t_, t_), b, accumulate=True)
    return B

def dc_balance_penalty(pred, data, device):
    # Per-graph DC mismatch: || B @ theta - P_net ||^2, averaged over buses, for our physics loss.
    total_penalty = torch.tensor(0.0, device=device, requires_grad=True)
    batch_vec = data.batch
    n_graphs = int(batch_vec.max().item()) + 1

    for g in range(n_graphs):
        mask = (batch_vec == g)
        n_bus = mask.sum().item()
        if n_bus < 2:
            continue

        ei = data.edge_index[:, batch_vec[data.edge_index[0]] == g]
        local_map = torch.full((mask.shape[0],), -1, dtype=torch.long, device=device)
        local_map[mask] = torch.arange(n_bus, device=device)
        ei_local  = local_map[ei]
        ea = data.edge_attr[batch_vec[data.edge_index[0]] == g]
        p_pred = pred[mask]
        theta = p_pred[:, 1] * np.pi   # va_norm → radians
        p_gen_norm = p_pred[:, 2]
        p_load_norm = data.x[mask][:, 0]
        p_net = p_gen_norm - p_load_norm

        B = build_susceptance_matrix(ei_local, ea, n_bus, device)
        mismatch = B @ theta - p_net
        # Skip index 0 (slack bus)
        penalty_g = (mismatch[1:] ** 2).mean()
        total_penalty = total_penalty + penalty_g

    return total_penalty / max(n_graphs, 1)

def opf_loss(pred, data, lam, device):
    mse = F.mse_loss(pred, data.y)
    phys = dc_balance_penalty(pred, data, device)
    return mse + lam * phys, mse.item(), phys.item()

print(f"Loss defined. lambda_physics = {CONFIG['lambda_physics']}")

## Training

All three models are trained sequentially with the same setup:
- **Optimiser:** Adam, lr=1e-3, weight decay=1e-5
- **Schedule:** cosine annealing over 150 epochs (eta_min=1e-6)
- **Early stopping:** patience of 15 epochs on total validation loss
- **Gradient clipping:** max norm 1.0 - the physics penalty can produce large gradients in early epochs
- **Checkpointing:** best weights saved per architecture, reloaded at the end

In [ ]:
def train_epoch(model, loader, optimiser, cfg, device):
    model.train()
    total_loss = total_mse = total_phys = 0.0
    for data in loader:
        data = data.to(device)
        optimiser.zero_grad()
        pred = model(data)
        loss, mse, phys = opf_loss(pred, data, cfg['lambda_physics'], device)
        loss.backward()
        if cfg['grad_clip'] > 0:
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
        optimiser.step()
        total_loss += loss.item()
        total_mse += mse
        total_phys += phys
    n = len(loader)
    return total_loss / n, total_mse / n, total_phys / n

@torch.no_grad()
def val_epoch(model, loader, cfg, device):
    model.eval()
    total_loss = total_mse = 0.0
    for data in loader:
        data = data.to(device)
        pred = model(data)
        loss, mse, _ = opf_loss(pred, data, cfg['lambda_physics'], device)
        total_loss += loss.item()
        total_mse  += mse
    n = len(loader)
    return total_loss / n, total_mse / n

def train_model(model_name, cfg, train_loader, val_loader, device):
    print(f'\nTraining {model_name}')
    model = build_model(model_name, cfg).to(device)
    opt = Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched = CosineAnnealingLR(opt, T_max=cfg['epochs'], eta_min=1e-6)

    best_val = float('inf')
    patience_c = 0
    history = []
    ckpt_path = cfg['checkpoint_dir'] / f'{model_name}_best.pt'

    pbar = tqdm(range(1, cfg['epochs'] + 1), desc=model_name, leave=True)
    for epoch in pbar:
        tr_loss, tr_mse, tr_phys = train_epoch(model, train_loader, opt, cfg, device)
        va_loss, va_mse = val_epoch(model, val_loader, cfg, device)
        sched.step()

        history.append({
            'epoch': epoch,
            'train_loss': tr_loss, 'train_mse': tr_mse, 'train_phys': tr_phys,
            'val_loss':   va_loss, 'val_mse':   va_mse,
        })

        pbar.set_postfix({
            'tr':  f'{tr_loss:.4f}',
            'va':  f'{va_loss:.4f}',
            'lr':  f'{sched.get_last_lr()[0]:.2e}',
        })

        if va_loss < best_val:
            best_val   = va_loss
            patience_c = 0
            torch.save({
                'state_dict': model.state_dict(),
                'epoch':      epoch,
                'val_loss':   va_loss,
                'model_name': model_name,
            }, ckpt_path)
        else:
            patience_c += 1
            if patience_c >= cfg['patience']:
                print(f'Early stop at epoch {epoch} (best val={best_val:.5f})')
                break

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    print(f'  Best val loss: {best_val:.5f} (epoch {ckpt["epoch"]})')
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(cfg['results_dir'] / f'{model_name}_history.csv', index=False)
    return model, hist_df

trained_models = {}
training_histories = {}

for arch in ['GCN', 'GraphSAGE', 'GAT']:
    model, hist = train_model(arch, CONFIG, train_loader, val_loader, DEVICE)
    trained_models[arch] = model
    training_histories[arch] = hist

print('All models trained.')

## Hybrid Solver (Warm-Start)

This is the actual point of our work. Per test scenario:

1. Run GNN inference → predicted `(vm_pu, va_degree, p_gen)`
2. Inject predictions into the pandapower network as initial values for NR iterative solver.
3. Run Newton-Raphson with `init='results'` - pandapower picks up the injected state
4. Record iteration count via `net._ppc['iterations']` (the internal NR counter, not an estimate)
5. Repeat from cold start (`init='flat'`, V=1, θ=0) as the baseline
6. Log feasibility, iteration delta, and wall-clock runtime for both

The GNN predicts a close value whihc is fed to the NR, which performs a few iterations, to get to the optimal solution. On small, well-conditioned cases like the IEEE 14/30/118-bus systems, flat-start NR already converges in 3–5 iterations, so there's limited headroom here. It is likely that there will be noticeable differences on a larger scale.

In [ ]:
def build_net_from_npz(npz_path):
    # Reconstructing the pandapower network for a saved scenario. We reload the base case 
    # and re-apply the saved Pd/Qd values by denormalising using total_load_mw from meta.
    case_name = Path(npz_path).parent.name
    net = load_base_network(case_name)
    d = np.load(npz_path, allow_pickle=True)
    meta = json.loads(str(d['meta'][0]))
    bf = d['bus_features']  # col0=Pd_norm, col1=Qd_norm
    total_load = meta['total_load_mw']
    Pd_mw = bf[:, 0] * total_load
    Qd_mvar = bf[:, 1] * total_load
    bmap = {bid: i for i, bid in enumerate(net.bus.index)}
    for lid, lrow in net.load.iterrows():
        bi = bmap[lrow.bus]
        net.load.at[lid, 'p_mw'] = float(Pd_mw[bi])
        net.load.at[lid, 'q_mvar'] = float(Qd_mvar[bi])
    return net, bmap


def run_cold_start(net_fresh):
    # NR from flat start (V=1, theta=0). Standard baseline for cold start.
    net = copy.deepcopy(net_fresh)
    t0 = time.time()
    try:
        pp.runpp(net, algorithm='nr', init='flat',
                 max_iteration=CONFIG['max_nr_iterations'],
                 tolerance_mva=CONFIG['nr_tolerance'],
                 calculate_voltage_angles=True,
                 numba=False)
    except Exception:
        return False, CONFIG['max_nr_iterations'], time.time() - t0, None
    iters = int(net._ppc.get('iterations', CONFIG['max_nr_iterations']))
    vm = net.res_bus.vm_pu.values.copy() if net.converged else None
    return net.converged, iters, time.time() - t0, vm

def run_warm_start(net_fresh, gnn_vm, gnn_va_deg, gnn_pg, bmap):
    # NR warm-started from GNN predictions. Pandapower uses net.res_bus as the initial state when init='results'.
    net = copy.deepcopy(net_fresh)

    if not hasattr(net, 'res_bus') or net.res_bus is None or len(net.res_bus) == 0:
        try:
            pp.runpp(net, init='flat', max_iteration=2, numba=False)
        except Exception:
            pass

    if hasattr(net, 'res_bus') and net.res_bus is not None:
        for bus_id, i in bmap.items():
            if bus_id in net.res_bus.index and i < len(gnn_vm):
                net.res_bus.at[bus_id, 'vm_pu'] = float(np.clip(gnn_vm[i], 0.7, 1.3))
                net.res_bus.at[bus_id, 'va_degree'] = float(gnn_va_deg[i])

    for gidx, grow in net.gen.iterrows():
        bi = bmap.get(grow.bus, 0)
        if bi >= len(gnn_pg): continue
        p_min = float(grow.get('min_p_mw', 0.0))
        p_max = float(grow.get('max_p_mw', grow.get('p_mw', 100.0)))
        if np.isnan(p_max) or p_max <= 0: p_max = 100.0
        net.gen.at[gidx, 'p_mw'] = float(np.clip(gnn_pg[bi], max(p_min, 0.0), p_max))

    t0 = time.time()
    try:
        pp.runpp(net, algorithm='nr', init='results',
                 max_iteration=CONFIG['max_nr_iterations'],
                 tolerance_mva=CONFIG['nr_tolerance'],
                 calculate_voltage_angles=True,
                 numba=False)
    except Exception:
        return False, CONFIG['max_nr_iterations'], time.time() - t0, None
    iters = int(net._ppc.get('iterations', CONFIG['max_nr_iterations']))
    vm = net.res_bus.vm_pu.values.copy() if net.converged else None
    return net.converged, iters, time.time() - t0, vm

@torch.no_grad()
def gnn_inference(model, npz_path, device):
    d = np.load(npz_path, allow_pickle=True)
    meta = json.loads(str(d['meta'][0]))
    x = torch.tensor(d['bus_features'], dtype=torch.float32).to(device)
    ei = torch.tensor(d['edge_index'], dtype=torch.long).to(device)
    ea = torch.tensor(d['edge_features'], dtype=torch.float32).to(device)
    data = Data(x=x, edge_index=ei, edge_attr=ea)
    data.batch = torch.zeros(x.shape[0], dtype=torch.long, device=device)
    model.eval()
    pred = model(data).cpu().numpy()   # (N, 3)
    vm_pu = pred[:, 0]
    va_deg = pred[:, 1] * 180.0    # va_norm → degrees
    total_gen = meta['total_load_mw'] * 1.05
    p_gen_mw = pred[:, 2] * total_gen
    return vm_pu, va_deg, p_gen_mw, meta

In [ ]:
def evaluate_hybrid(trained_models, test_ds, cfg, device, n_max=None):
    records = []
    files = test_ds.files[:n_max] if n_max else test_ds.files

    for npz_path in tqdm(files, desc='Hybrid evaluation'):
        case_name = Path(npz_path).parent.name
        try:
            net_fresh, bmap = build_net_from_npz(npz_path)
        except Exception:
            continue

        # These are the common cold start values for comparison
        cold_ok, cold_iters, cold_rt, cold_vm = run_cold_start(net_fresh)

        for model_name, model in trained_models.items():
            try:
                gnn_vm, gnn_va, gnn_pg, meta = gnn_inference(model, npz_path, device)
            except Exception:
                continue

            warm_ok, warm_iters, warm_rt, warm_vm = run_warm_start(
                net_fresh, gnn_vm, gnn_va, gnn_pg, bmap
            )

            d = np.load(npz_path, allow_pickle=True)
            y_true = d['targets']
            mape_vm = float(np.mean(
                np.abs((gnn_vm - y_true[:, 0]) / (np.abs(y_true[:, 0]) + 1e-6))
            ) * 100)

            v_viol = 0
            if warm_ok and warm_vm is not None:
                v_viol = int(np.sum(
                    (warm_vm < cfg['v_min_pu']) | (warm_vm > cfg['v_max_pu'])
                ))

            iter_reduction = (
                (cold_iters - warm_iters) / max(cold_iters, 1) * 100
                if cold_ok else 0.0
            )
            speedup = cold_rt / max(warm_rt, 1e-9) if cold_ok and warm_ok else 1.0

            records.append({
                'case':               case_name,
                'n_bus':              meta['n_bus'],
                'model':              model_name,
                'cold_converged':     cold_ok,
                'cold_iters':         cold_iters,
                'cold_rt_s':          cold_rt,
                'warm_converged':     warm_ok,
                'warm_iters':         warm_iters,
                'warm_rt_s':          warm_rt,
                'iter_reduction_pct': iter_reduction,
                'rt_speedup':         speedup,
                'gnn_mape_vm_pct':    mape_vm,
                'v_violations_warm':  v_viol,
            })

    df = pd.DataFrame(records)
    df.to_csv(cfg['results_dir'] / 'hybrid_solver_results.csv', index=False)
    print(f'Hybrid evaluation done: {len(df)} records')
    return df

hybrid_df = evaluate_hybrid(trained_models, test_ds, CONFIG, DEVICE)
hybrid_df.head()

## Evaluation

Metrics across all three architectures:
- **GNN accuracy:** MAPE and MAE on `vm_pu`, `va_norm`, `p_gen_norm` vs OPF ground truth
- **NR iterations:** mean warm-start vs cold-start iteration counts
- **Feasibility rate:** fraction of warm-started runs that converge
- **Runtime:** wall-clock speedup ratio

Note on MAPE: it's only reported for `vm_pu` (denominator bounded away from zero at ≥ 0.9 p.u.). For `va_norm` and `p_gen_norm`, MAPE is meaningless because the denominator is near-zero at many buses,exploding the value. Hence, MAE and RMSE are used instead.

In [ ]:
@torch.no_grad()
def compute_gnn_metrics(model, loader, device):
    model.eval()
    all_pred, all_true = [], []
    for data in loader:
        data = data.to(device)
        pred = model(data).cpu().numpy()
        true = data.y.cpu().numpy()
        all_pred.append(pred)
        all_true.append(true)

    pred = np.vstack(all_pred)   # (total_nodes, 3) since we are predicting 3 values
    true = np.vstack(all_true)

    metrics = {}
    for j, nm in enumerate(['vm_pu', 'va_norm', 'p_gen_norm']):
        p, t = pred[:, j], true[:, j]
        entry = {
            'mae': float(np.mean(np.abs(p - t))),
            'rmse': float(np.sqrt(np.mean((p - t) ** 2))),
        }
        if nm == 'vm_pu':
            entry['mape_%'] = float(np.mean(np.abs((p - t) / (np.abs(t) + 1e-6))) * 100)
        metrics[nm] = entry
    return metrics

acc_records = []
for name, model in trained_models.items():
    m = compute_gnn_metrics(model, test_loader, DEVICE)
    row = {'Model': name}
    row['vm_pu_MAPE%'] = round(m['vm_pu']['mape_%'], 3)
    row['vm_pu_MAE'] = round(m['vm_pu']['mae'],    5)
    row['vm_pu_RMSE'] = round(m['vm_pu']['rmse'],   5)
    row['va_norm_MAE'] = round(m['va_norm']['mae'],  5)
    row['va_norm_RMSE'] = round(m['va_norm']['rmse'], 5)
    row['p_gen_MAE'] = round(m['p_gen_norm']['mae'],  5)
    row['p_gen_RMSE'] = round(m['p_gen_norm']['rmse'], 5)
    acc_records.append(row)

acc_df = pd.DataFrame(acc_records)
acc_df.to_csv(CONFIG['results_dir'] / 'gnn_accuracy.csv', index=False)

print('GNN prediction accuracy on test set')
print(acc_df.to_string(index=False))

In [ ]:
converged = hybrid_df[hybrid_df['warm_converged']]

summary = (
    converged.groupby('model')
    .agg(
        mean_cold_iters = ('cold_iters', 'mean'),
        mean_warm_iters = ('warm_iters', 'mean'),
        iter_reduction_pct = ('iter_reduction_pct', 'mean'),
        mean_speedup = ('rt_speedup', 'mean'),
        mean_gnn_mape_vm = ('gnn_mape_vm_pct', 'mean'),
        mean_v_violations = ('v_violations_warm', 'mean'),
        n_scenarios = ('warm_converged', 'count'),
    )
    .round(3)
    .reset_index()
)

# Feasibility over all test scenarios
feas = hybrid_df.groupby('model')['warm_converged'].mean().reset_index()
feas.columns = ['model', 'feasibility_rate_all']
summary = summary.merge(feas, on='model')

summary.to_csv(CONFIG['results_dir'] / 'hybrid_summary.csv', index=False)
print('Hybrid solver summary:')
print(summary.to_string(index=False))

In [ ]:
size_summary = (
    converged.groupby(['model', 'case'])
    .agg(
        mean_cold_iters = ('cold_iters', 'mean'),
        mean_warm_iters = ('warm_iters', 'mean'),
        iter_red_pct = ('iter_reduction_pct', 'mean'),
        n_bus = ('n_bus', 'first'),
        count = ('warm_converged', 'count'),
    )
    .round(2)
    .reset_index()
    .sort_values(['model', 'n_bus'])
)

size_summary.to_csv(CONFIG['results_dir'] / 'size_breakdown.csv', index=False)
print('Per-case breakdown:')
print(size_summary.to_string(index=False))

## Figures

In [ ]:
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
PALETTE = {'GCN': '#4C72B0', 'GraphSAGE': '#DD8452', 'GAT': '#55A868'}
PDIR = CONFIG['plots_dir']

In [ ]:
# Figure 1: Training loss curves
# Log scale on y because the physics penalty starts enormous (B·θ before the GNN has learned reasonable angles) and collapses 
# within a few epochs. Linear scale compresses all the interesting post-convergence behaviour.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, (name, hist) in zip(axes, training_histories.items()):
    ax.plot(hist['epoch'], hist['train_loss'], color=PALETTE[name], label='Train', linewidth=2)
    ax.plot(hist['epoch'], hist['val_loss'], color=PALETTE[name], label='Val', linewidth=2, linestyle='--')
    ax.fill_between(hist['epoch'], hist['train_loss'], hist['val_loss'], alpha=0.08, color=PALETTE[name])
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (log scale)')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3, which='both')

fig.suptitle('Training and Validation Loss - MSE + Physics Penalty (log scale)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PDIR / 'fig1_loss_curves.png')
plt.show()

In [ ]:
# Figure 2: Prediction accuracy by target and architecture
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
models = [r['Model'] for r in acc_records]
colors = [PALETTE[m] for m in models]

for ax, (key, title, ylabel) in zip(axes, [
    ('vm_pu_MAPE%', 'Voltage Magnitude (vm_pu) - MAPE', 'MAPE (%)'),
    ('va_norm_MAE', 'Voltage Angle (va_norm) - MAE', 'MAE (p.u.)'),
    ('p_gen_MAE', 'Generator Power (p_gen) - MAE', 'MAE (p.u.)'),
]):
    vals = [r[key] for r in acc_records]
    bars = ax.bar(models, vals, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
    fmt  = '{:.2f}%' if 'MAPE' in key else '{:.4f}'
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02, fmt.format(v), ha='center', va='bottom', fontsize=9)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(vals) * 1.4)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('GNN Prediction Accuracy vs OPF Ground Truth (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PDIR / 'fig2_gnn_accuracy.png')
plt.show()

In [ ]:
# Figure 3: NR iteration comparison - warm vs cold
# On IEEE 14/30/118-bus systems NR from flat start already converges in 3-5 iterations, so the iteration reduction is modest. The more important 
# result is feasibility - the warm start doesn't hurt convergence of any scenario in the considered cases, which is extremely important for any 
# form of real-world appliactions.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x  = np.arange(len(summary))
w  = 0.35
b1 = ax.bar(x - w/2, summary['mean_cold_iters'], w, label='Cold start (V=1, θ=0)', color='#c44e52', alpha=0.85)
b2 = ax.bar(x + w/2, summary['mean_warm_iters'], w, label='Warm start (GNN init)', color='#4c72b0', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(summary['model'])
ax.set_ylabel('Mean NR Iterations')
ax.set_title('Newton-Raphson Iterations: Warm vs Cold Start')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{bar.get_height():.1f}', ha='center', fontsize=9)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{bar.get_height():.1f}', ha='center', fontsize=9)

ax2 = axes[1]
for name, grp in size_summary.groupby('model'):
    ax2.plot(grp['n_bus'], grp['iter_red_pct'], marker='o', label=name, color=PALETTE[name], linewidth=2)
ax2.set_xlabel('Network Size (buses)')
ax2.set_ylabel('Iteration reduction vs cold start (%)')
ax2.set_title('Iteration Delta by Network Size\n(negative = warm start needs more iters; expected for small cases)')
ax2.axhline(0, linestyle='--', color='grey', alpha=0.7, label='Parity')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

fig.suptitle('GNN Warm-Start: Solver Reliability and Iteration Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PDIR / 'fig3_iteration_comparison.png')
plt.show()

In [ ]:
# Figure 4: Accuracy vs runtime
fig, ax = plt.subplots(figsize=(8, 6))

for rec in acc_records:
    name = rec['Model']
    mape = rec['vm_pu_MAPE%']
    rt = hybrid_df[hybrid_df['model'] == name]['warm_rt_s'].mean()
    ax.scatter(rt, mape, s=180, color=PALETTE[name], label=f'{name}', zorder=5)
    ax.annotate(f'  {name}', (rt, mape), fontsize=10)

cold_rt = hybrid_df['cold_rt_s'].mean()
ax.axvline(cold_rt, linestyle='--', color='red', alpha=0.6, label=f'Cold-start NR avg ({cold_rt:.3f}s)')

ax.set_xlabel('Mean runtime (seconds)')
ax.set_ylabel('vm_pu MAPE (%)')
ax.set_title('Accuracy vs Runtime - lower-left is better')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PDIR / 'fig4_pareto_frontier.png')
plt.show()

In [ ]:
# Figure 5: Runtime scalability across network sizes (log-log)
fig, ax = plt.subplots(figsize=(9, 5))

cold_by_size = (
    hybrid_df[hybrid_df['cold_converged']]
    .groupby('n_bus')['cold_rt_s'].mean()
    .reset_index()
)
ax.plot(cold_by_size['n_bus'], cold_by_size['cold_rt_s'], color='red', marker='s', linewidth=2.5, markersize=7, label='Cold-start NR', zorder=4)

for name, grp in hybrid_df[hybrid_df['warm_converged']].groupby('model'):
    ws = grp.groupby('n_bus')['warm_rt_s'].mean().reset_index()
    ax.plot(ws['n_bus'], ws['warm_rt_s'], color=PALETTE[name], marker='o', linewidth=2, markersize=6, label=f'{name} (warm)', linestyle='--')

ax.set_xlabel('Network size (buses)')
ax.set_ylabel('Mean runtime (seconds)')
ax.set_title('Runtime Scalability: Cold Start vs GNN Warm Start')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PDIR / 'fig5_scalability.png')
plt.show()